# 02 · Vectorizer Comparison — BoW vs TF-IDF vs BM25

The lab's central empirical question: *when* do TF-IDF and BM25 beat raw counts, and on which dataset is the gap largest?

> Run `make data` first. This notebook trains lightweight models inline; for tracked runs use `make train`.


In [ ]:
import pandas as pd
from nlp_system.data.make_dataset import load_split
from nlp_system.models.train import build_pipeline, _evaluate
from nlp_system.pipeline.nltk_setup import ensure_nltk
ensure_nltk()


## Helper: sweep the three vectorizers on a dataset


In [ ]:
def sweep(dataset, sample=20000):
    tr = load_split(dataset, 'train').sample(min(sample, 999999), random_state=42)
    va = load_split(dataset, 'val')
    rows = []
    for m in ['bow', 'tfidf', 'bm25']:
        pipe = build_pipeline(dataset, m, max_features=50000)
        pipe.fit(tr['text'].tolist(), tr['label'].values)
        mt = _evaluate(pipe, va['text'].tolist(), va['label'].values)
        rows.append({'vectorizer': m, 'f1': round(mt.f1,4), 'roc_auc': round(mt.roc_auc,4)})
    return pd.DataFrame(rows)


## Amazon Fine Food


In [ ]:
amz = sweep('amazon_fine_food'); amz


## Sentiment140


In [ ]:
s140 = sweep('sentiment140'); s140


## Side-by-side

Expected pattern: BM25 ≥ TF-IDF > BoW, with the BM25 edge larger on long Amazon reviews where term-frequency saturation and length normalization bite.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,2, figsize=(11,4))
amz.set_index('vectorizer')['f1'].plot.bar(ax=ax[0], title='Amazon F1', ylim=(0.7,1.0))
s140.set_index('vectorizer')['f1'].plot.bar(ax=ax[1], title='Sentiment140 F1', ylim=(0.6,0.9))
plt.tight_layout(); plt.show()


### Interpretation (fill in after running)

- Gap BoW→TF-IDF on Amazon: …
- Gap TF-IDF→BM25 on Amazon: …
- Same gaps on Sentiment140: …
- Conclusion on *when* the fancier vectorizers earn their keep: …
